# WILDTRACK Dataset Sample
Uses the WILDTRACK dataset from Kaggle (already mounted), generates MP4 videos
from frame images, and packages a sample for local testing.

**Dataset**: [Large Scale Multi-Camera Detection Dataset](https://www.kaggle.com/datasets/aryashah2k/large-scale-multicamera-detection-dataset)

**Output**: `wildtrack_sample.zip` with:
- All 7 cameras as MP4 video
- Ground truth annotation JSON files
- Camera calibration files

In [ ]:
import os, shutil
from pathlib import Path

WORK = Path("/kaggle/working")

# Kaggle mounts attached datasets at /kaggle/input/<dataset-slug>
WT_INPUT = Path("/kaggle/input/large-scale-multicamera-detection-dataset")
assert WT_INPUT.exists(), f"Dataset not found at {WT_INPUT}. Attach it in notebook settings."

print("WILDTRACK dataset contents:")
for item in sorted(WT_INPUT.iterdir()):
    if item.is_dir():
        n = len(list(item.rglob("*")))
        print(f"  {item.name}/ ({n} files)")
    else:
        print(f"  {item.name} ({item.stat().st_size/(1024**2):.1f} MB)")

In [ ]:
!pip install -q opencv-python-headless 2>&1 | tail -3

## 1. Verify Dataset Structure

In [ ]:
# Find Image_subsets - might be at root or inside a subfolder
img_root = None
for candidate in [WT_INPUT / "Image_subsets", *WT_INPUT.glob("*/Image_subsets")]:
    if candidate.exists():
        img_root = candidate
        break

# Find annotations
ann_root = None
for candidate in [WT_INPUT / "annotations_positions", *WT_INPUT.glob("*/annotations_positions")]:
    if candidate.exists():
        ann_root = candidate
        break

# Find calibrations
cal_root = None
for candidate in [WT_INPUT / "calibrations", *WT_INPUT.glob("*/calibrations")]:
    if candidate.exists():
        cal_root = candidate
        break

print(f"Image_subsets: {img_root}")
print(f"annotations:   {ann_root}")
print(f"calibrations:  {cal_root}")

assert img_root is not None, "Image_subsets not found!"

cams = sorted(d for d in img_root.iterdir() if d.is_dir())
print(f"\nFound {len(cams)} cameras:")
for cam_dir in cams:
    frames = list(cam_dir.glob("*.png"))
    print(f"  {cam_dir.name}: {len(frames)} frames")

## 2. Generate MP4 Videos from Frame Sequences
WILDTRACK provides PNG frames (sampled at 2fps from a 60fps stream).
We encode them as MP4 so our pipeline's video-based ingestion can use them.

In [ ]:
import cv2

videos_dir = WORK / "wildtrack_videos"
videos_dir.mkdir(exist_ok=True)

for cam_dir in cams:
    frames = sorted(cam_dir.glob("*.png"))
    if not frames:
        continue
    
    video_path = videos_dir / f"{cam_dir.name}.mp4"
    if video_path.exists():
        print(f"  {video_path.name} already exists, skipping")
        continue
    
    sample = cv2.imread(str(frames[0]))
    if sample is None:
        print(f"  WARNING: Could not read {frames[0]}")
        continue
    h, w = sample.shape[:2]
    
    # WILDTRACK is 2 fps
    writer = cv2.VideoWriter(
        str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), 2.0, (w, h)
    )
    for fp in frames:
        img = cv2.imread(str(fp))
        if img is not None:
            writer.write(img)
    writer.release()
    
    size_mb = video_path.stat().st_size / (1024**2)
    print(f"  {video_path.name}: {len(frames)} frames, {w}x{h}, {size_mb:.0f} MB")

## 3. Build Sample Package
Package all 7 cameras (MP4 videos + annotations + calibrations) for download.

In [ ]:
SAMPLE_DIR = WORK / "wt_sample"
if SAMPLE_DIR.exists():
    shutil.rmtree(SAMPLE_DIR)

wt_out = SAMPLE_DIR / "wildtrack"

# Copy videos
dest_vid = wt_out / "videos"
dest_vid.mkdir(parents=True, exist_ok=True)
for v in sorted(videos_dir.glob("*.mp4")):
    shutil.copy2(v, dest_vid / v.name)
    print(f"  {v.name}: {v.stat().st_size/(1024**2):.0f} MB")

# Copy annotations
if ann_root:
    dest_ann = wt_out / "annotations_positions"
    shutil.copytree(ann_root, dest_ann)
    n = len(list(dest_ann.glob("*.json")))
    print(f"  annotations: {n} JSON files")

# Copy calibrations
if cal_root:
    dest_cal = wt_out / "calibrations"
    shutil.copytree(cal_root, dest_cal)
    n = len(list(dest_cal.rglob("*")))
    print(f"  calibrations: {n} files")

total = sum(f.stat().st_size for f in SAMPLE_DIR.rglob("*") if f.is_file())
print(f"\nTotal sample size: {total/(1024**3):.2f} GB")

In [ ]:
import zipfile

ZIP_PATH = WORK / "wildtrack_sample.zip"

print("Creating zip...")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_STORED) as zf:
    for f in sorted(SAMPLE_DIR.rglob("*")):
        if f.is_file():
            arcname = str(f.relative_to(SAMPLE_DIR))
            zf.write(f, arcname)

print(f"\nDone: {ZIP_PATH.name} ({ZIP_PATH.stat().st_size/(1024**3):.2f} GB)")
print("Download from the Output tab on the right.")
print("\nExtract to data/raw/ in your local project:")
print("  data/raw/wildtrack/videos/C1.mp4 ...")
print("  data/raw/wildtrack/annotations_positions/*.json")
print("  data/raw/wildtrack/calibrations/...")

In [ ]:
# Clean up intermediate files to save space
shutil.rmtree(videos_dir, ignore_errors=True)
shutil.rmtree(SAMPLE_DIR, ignore_errors=True)

# Summary
print("=" * 60)
print("WILDTRACK SAMPLE")
print("=" * 60)
print(f"Output: {ZIP_PATH.name} ({ZIP_PATH.stat().st_size/(1024**3):.2f} GB)")
print(f"Cameras: {len(cams)}")
print("Contents: MP4 videos + JSON annotations + calibrations")